In [1]:
import pandas as pd 

In [2]:
df=pd.read_csv("data/IMDB Dataset.csv")

In [3]:
df.head()
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

## Preprocessing

In [7]:
# Text cleaning 
import re 
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def clean_text(text):
    text=text.lower()
    text=re.sub(r"http\S+","",text)
    text=re.sub(r"<.*?>","",text)
    text=re.sub(r"[^a-zA-Z\s]","",text)

    words=text.split()
    stop_words=set(stopwords.words("english"))
    filtered_words=[w for w in words if w not in stop_words]
    return " ".join(filtered_words)

df["review"]=df["review"].apply(clean_text)

In [8]:
df.head()

,review,sentiment
0,one reviewers mentioned watching oz episode yo...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [9]:
from collections import Counter
import torch 
from torch.utils.data import DataLoader,TensorDataset

#1 count word frequencies across the entire dataset 
all_words=[]
for review in df['review']:
    all_words.extend(review.split())

word_counts=Counter(all_words)

# 2. Select top 10000 most common words 
MAX_VOCAB_SIZE=10000
most_common=word_counts.most_common(MAX_VOCAB_SIZE)

#3 Create vocabulary mapping (0: pad,1:Unknown)
vocab={"<PAD>":0,"<UNK>":1}

for word ,_ in most_common:
    vocab[word]=len(vocab)

print(f"Vocabulary Size: {len(vocab)}")

Vocabulary Size: 10002


## Stemming

In [10]:
from nltk.stem import PorterStemmer

def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    tokens=word_tokenize(text)

    for token in tokens:
        stemmed_token=ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

In [11]:
df["review"]=df["review"].apply(stemming)

In [12]:
df.head()

,review,sentiment
0,one review mention watch oz episod youll hook ...,positive
1,wonder littl product film techniqu unassum old...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


In [13]:
# 7. Encoding 
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df["sentiment"]=le.fit_transform(df["sentiment"])

In [14]:
y=df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [15]:
# 8 Vectorization
df.head()

,review,sentiment
0,one review mention watch oz episod youll hook ...,1
1,wonder littl product film techniqu unassum old...,1
2,thought wonder way spend time hot summer weeke...,1
3,basic there famili littl boy jake think there ...,0
4,petter mattei love time money visual stun film...,1


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df["review"])

In [17]:
x

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4036037 stored elements and shape (49582, 5000)>

## Dataset and Dataloaders

In [18]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(
    x,y,test_size=0.2,random_state=42
)

In [19]:
x_train.shape

(39665, 5000)

In [20]:
x_test.shape

(9917, 5000)

In [28]:
x_test=x_test.toarray()

In [29]:
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [30]:
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

In [31]:
#Build our RNN
import torch.nn as nn
import torch.optim as optim

In [32]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layers=1):
        super(RNN,self).__init__()

        self.hidden_size=hidden_size
        self.num_layers=num_layers

        #RNN layer
        self.rnn=nn.RNN(input_size,hidden_size,num_layers,batch_first=True)# Batch first changes the shape of input

        # fully connected layer 
        self.fc=nn.Linear(hidden_size,1)

    def forward(self,x):
        # Initialize hidden 
        h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)

        out,_=self.rnn(x,h0) # 1st value is hidden state of all the timesteps [batch,seq_len,hidden_size]
        # 2nd value final hidden state of last timestep 

        out=self.fc(out[:,-1,:])
        return out

In [33]:
input_size=x_train.shape[1]
model=RNN(input_size)

criterion=nn.BCELoss()
optimizer=optim.AdamW(model.parameters())

# Training the RNN

In [34]:
epochs=10

for epoch in range(epochs):
    model.train()
    for xb,yb in train_loader:
        optimizer.zero_grad()
        xb=xb.unsqueeze(1) # add 1 dimension
        outputs=model(xb) # expects 3d 
        outputs=torch.sigmoid(outputs.squeeze()) #convert to single value
        loss=criterion(outputs,yb)
        loss.backward()
        optimizer.step()
    
    print(f"Epoch:{epoch+1}/{epochs} and loss:{loss.item()}")

Epoch:1/10 and loss:0.15775007009506226
Epoch:2/10 and loss:0.17549875378608704
Epoch:3/10 and loss:0.1949201077222824
Epoch:4/10 and loss:0.214102640748024
Epoch:5/10 and loss:0.13234269618988037
Epoch:6/10 and loss:0.2248428761959076
Epoch:7/10 and loss:0.26430341601371765
Epoch:8/10 and loss:0.2714236378669739
Epoch:9/10 and loss:0.30917656421661377
Epoch:10/10 and loss:0.08623847365379333


In [35]:
#Evaluation 
model.eval()
with torch.no_grad():
    correct_vals=0
    total_vals=0

    for xb,yb in test_loader:
        xb=xb.unsqueeze(1)
        outputs=model(xb)
        predicted=(torch.sigmoid(outputs.squeeze())>0.5).float()
        total_vals+=yb.size(0)
        correct_vals+=(predicted==yb).sum().item()

    print(f"Accuracy:{(correct_vals/total_vals)*100}")

Accuracy:86.8407784612282


In [ ]:
def predict_sentiment(review_text):
    model.eval()
    
    # 1. Clean the text
    cleaned = clean_text(review_text)
    
    # 2. Stem the cleaned text 
    stemmed = stemming(cleaned)
    
    # 3. Vectorize using TF-IDF
    vectorized = tf.transform([stemmed]).toarray()
    
    # 4. Convert to Tensor & add sequence dimension (1, 1, 5000)
    tensor_input = torch.from_numpy(vectorized).float().unsqueeze(1)
    
    # 5. Predict
    with torch.no_grad():
        output = model(tensor_input)
        prob = torch.sigmoid(output.squeeze()).item()
        
    sentiment = "Positive " if prob >= 0.5 else "Negative"
    print(f"Review: '{review_text}'")
    print(f"Prediction: {sentiment} (Confidence: {prob * 100:.2f}%)\n")


In [39]:
predict_sentiment("This movie was fantastic! Outstanding performance by the lead actor.")
predict_sentiment("Terrible movie. Horrible pacing and super boring story.")

Review: 'This movie was fantastic! Outstanding performance by the lead actor.'
Prediction: Positive 😁 (Confidence: 99.94%)

Review: 'Terrible movie. Horrible pacing and super boring story.'
Prediction: Negative 😞 (Confidence: 0.00%)



In [ ]:
import os
import pickle
import torch

# 1. Ensure model directory exists
os.makedirs("model", exist_ok=True)

# 2. Save PyTorch Model weights (.pth)
model_path = "model/moviemood_rnn.pth"
torch.save(model.state_dict(), model_path)
print(f" Model weights saved to: {model_path}")

# 3. Save TF-IDF Vectorizer (.pkl)
vectorizer_path = "model/tfidf_vectorizer.pkl"
with open(vectorizer_path, "wb") as f:
    pickle.dump(tf, f)
print(f" TF-IDF Vectorizer saved to: {vectorizer_path}")


✅ Model weights saved to: model/moviemood_rnn.pth
✅ TF-IDF Vectorizer saved to: model/tfidf_vectorizer.pkl
